In [36]:
import requests
import xml.etree.ElementTree as ET
import polars as pl
import time

In [11]:
# Suchparameter
query = '"german"[Language] AND ("2025/01/01"[PDAT] : "2025/12/31"[PDAT])'
#alle Deutschen Veröffentlichungen des Jahres 2025
apiKey = "7e14fd2cb71ccbb3dade927e99df741a5009" #um auf 10 Veröffentlichungen zugreifen zu können
#retmax = 50 #return maximum (anzahl der Ergebnisse, die die API maximal zurückgeben soll)

#aufruf von esearch
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db" : "pubmed",
    "term" : query,
    "retmax" : retmax,
    "retmode" : "json",
    "api_key" : api_key
}
#auf PubMed nach dem Suchstring suchen und die ersten 50(retmax) einträge ausgeben

response = requests.get(url, params=params)
data = response.json()
pmids = data["esearchresult"]["idlist"]
print(f"{len(pmids)} PMIDs gefunden") #Ausgabe der gefundenen PMIDs
print(pmids)


50 PMIDs gefunden
['41973606', '41971571', '41971570', '41971569', '41971564', '41971561', '41921923', '41688089', '41586736', '41586734', '41586733', '41586732', '41586730', '41586729', '41586726', '41586725', '41586722', '41586721', '41586720', '41586717', '41586715', '41586712', '41586710', '41586709', '41586704', '41586700', '41586698', '41586446', '41586084', '41586082', '41584498', '41584495', '41583135', '41583133', '41583132', '41571265', '41569278', '41569277', '41569276', '41569275', '41569274', '41569273', '41569272', '41569250', '41569249', '41569248', '41569247', '41569246', '41569245', '41569244']


In [12]:
#efetch Aufruf Metadaten als XML laden
ids = " , ".join(pmids)
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
params = {
    "db" : "pubmed",
    "id" : ids,
    "retmode" : "xml",
    "api_key": apiKey
}
response = requests.get(url, params=params)
xmlData = response.text
print(xmlData[:500])  #Anzeigen der ersten 500 Zeichen


<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
<PubmedArticle><MedlineCitation Status="PubMed-not-MEDLINE" Owner="NLM"><PMID Version="1">41973606</PMID><DateCompleted><Year>2026</Year><Month>04</Month><Day>13</Day></DateCompleted><DateRevised><Year>2026</Year><Month>04</Month><Day>13</Day></DateRevised><Article PubModel="Electronic-eCollection"><Journal><ISS


In [28]:
#parsen der XML und bauen des Polars DataFrames
def parsePubmedXML(xmlText) :
    root = ET.fromstring(xmlText)
    articles = []
    for article in root.findall(".//PubmedArticle"):

            #PMID
        pmid = article.findtext(".//PMID", default="")

            #titel
        title = article.findtext(".//ArticleTitle", default="")
            #sollte kein Englischer Titel vorhanden sein, den Deutschen nehmen(woanders gespeichert)
        if not title or title == "[Not Available].":
            title = article.findtext(".//VernacularTitle", default="kein Titel verfügbar") #evtl schauen wie Titel noch gespeichert werden können

            #Autoren und Affiliationen
        authors = []
        affilliations = []

        for author in article.findall(".//Author"):
            last = author.findtext("LastName", default="")
            fore = author.findtext("ForeName", default="")
            #hier nochmal schauen
            if last:
                authors.append(f"{last} {fore}".strip())
            for affil in author.findall("AffiliationInfo/Affiliation"):
                if affil.text and affil.text not in affilliations:
                    affilliations.append(affil.text)

        #Erstautor
        firstAuthor = authors[0] if authors else ""

        #link
        link = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

        articles.append({
            "pmid" : pmid,
            "title" : title,
            "firstAuthor" : firstAuthor,
            "authors" : "; ".join(authors),
            "affiliations" : " | ".join(affilliations),
            "link": link
        })

    return articles

artikel = parsePubmedXML(xmlData)
print(f"{len(artikel)} Artikel geparsed")
print(artikel[0]) #ersten Artikel zur Kontrolle anzeigen

50 Artikel geparsed
{'pmid': '41973606', 'title': 'Mykotoxine – Bestimmung von Aflatoxinen, Ochratoxin\xa0A, freiem Ochratoxin\xa0', 'firstAuthor': 'Berger Marion', 'authors': 'Berger Marion; Deharde Max; Neuhoff Judith; Monien Bernhard; Siodlaczek Solveigh; Göen Thomas; Hartwig Andrea', 'affiliations': 'Bundesanstalt für Arbeitsschutz und Arbeitsmedizin (BAuA). Fachbereich\xa04 - Gefahrstoffe und biologische Arbeitsstoffe. Gruppe\xa04.2 - Medizinischer Arbeitsschutz. Biomonitoring Nöldnerstraße\xa040/42 10317 Berlin Deutschland. | Bundesinstitut für Risikobewertung. Fachgruppe\xa054 - Abt. Lebensmittelsicherheit Max-Dohrn-Straße\xa08-10 10589 Berlin Deutschland. | Friedrich-Alexander-Universität Erlangen-Nürnberg. Institut und Poliklinik für Arbeits-, Sozial- und Umweltmedizin Henkestraße 9-11 91054 Erlangen Deutschland. | Institut für Angewandte Biowissenschaften. Abteilung Lebensmittelchemie und Toxikologie. Karlsruher Institut für Technologie (KIT) Adenauerring 20a, Geb. 50.41 7613

In [20]:
#fehlerhebung zur überprüfung wie die Affiliationen ausgegeben werden

root = ET.fromstring(xmlData)
ersterArtikel = root.find(".//PubmedArticle")
for author in ersterArtikel.findall(".//Author"):
    print("Autor: ", author.findtext("LastName"))
    for affil in author.findall("AffiliationInfo/Affiliation"):
        print("  Affiliation:", affil.text)

Autor:  Berger
  Affiliation: Bundesanstalt für Arbeitsschutz und Arbeitsmedizin (BAuA). Fachbereich 4 - Gefahrstoffe und biologische Arbeitsstoffe. Gruppe 4.2 - Medizinischer Arbeitsschutz. Biomonitoring Nöldnerstraße 40/42 10317 Berlin Deutschland.
Autor:  Deharde
  Affiliation: Bundesanstalt für Arbeitsschutz und Arbeitsmedizin (BAuA). Fachbereich 4 - Gefahrstoffe und biologische Arbeitsstoffe. Gruppe 4.2 - Medizinischer Arbeitsschutz. Biomonitoring Nöldnerstraße 40/42 10317 Berlin Deutschland.
Autor:  Neuhoff
  Affiliation: Bundesanstalt für Arbeitsschutz und Arbeitsmedizin (BAuA). Fachbereich 4 - Gefahrstoffe und biologische Arbeitsstoffe. Gruppe 4.2 - Medizinischer Arbeitsschutz. Biomonitoring Nöldnerstraße 40/42 10317 Berlin Deutschland.
Autor:  Monien
  Affiliation: Bundesinstitut für Risikobewertung. Fachgruppe 54 - Abt. Lebensmittelsicherheit Max-Dohrn-Straße 8-10 10589 Berlin Deutschland.
Autor:  Siodlaczek
  Affiliation: Bundesinstitut für Risikobewertung. Fachgruppe 54 - A

In [31]:
# DataFrame bauen
df = pl.DataFrame(artikel)

#\xa0 aus xml bereinigen (geschütztes leerzeichen)
df = df.with_columns([
    pl.col("affiliations").str.replace_all("\xa0", " "),
    pl.col("title").str.replace_all("\xa0", " ")
])

print(df.shape)
df.head()
#df.filter(pl.col("title") != "[Not Available].").head()

(50, 6)


pmid,title,firstAuthor,authors,affiliations,link
str,str,str,str,str,str
"""41973606""","""Mykotoxine – Bestimmung von Af…","""Berger Marion""","""Berger Marion; Deharde Max; Ne…","""Bundesanstalt für Arbeitsschut…","""https://pubmed.ncbi.nlm.nih.go…"
"""41971571""","""Cadmium und seine anorganische…","""Hallier Ernst""","""Hallier Ernst; Drexler Hans; H…","""37136 Ebergötzen Deutschland. …","""https://pubmed.ncbi.nlm.nih.go…"
"""41971570""","""Tri-(2-ethylhexyl)trimellitat …","""Kuhlmann Laura""","""Kuhlmann Laura; Eckert Elisabe…","""Friedrich-Alexander-Universitä…","""https://pubmed.ncbi.nlm.nih.go…"
"""41971569""","""Glaswolle, Halbwertszeit < 40 …","""Hartwig Andrea""","""Hartwig Andrea""","""Institut für Angewandte Biowis…","""https://pubmed.ncbi.nlm.nih.go…"
"""41971564""","""5-Ethyl-3,7-dioxa-1-azabicyclo…","""Hartwig Andrea""","""Hartwig Andrea""","""Institut für Angewandte Biowis…","""https://pubmed.ncbi.nlm.nih.go…"


In [32]:
df.write_csv("pubmed_rohdaten.csv")
print("Gespeichert!")

Gespeichert!


In [33]:
#abfrage Gesamtzahl ohne Daten zu laden
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db" : "pubmed",
    "term" : query,
    "retmode" : "json",
    "api_key" : apiKey
}
response = requests.get(url, params=params)
data = response.json()
total = int(data["esearchresult"]["count"])
print(f"Gesamtanzahl Artikel: {total}")

Gesamtanzahl Artikel: 6505


In [40]:
#Alle PMIDs werden als Batches geholt

allePmids = []
batchSize = 500

for start in range (0, total, batchSize):
    params = {
        "db" : "pubmed",
        "term":query,
        "retmax": batchSize,
        "retstart" : start,
        "retmode" : "json",
        "api_key" : apiKey
    }

    response = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", params=params)
    data = response.json()
    pmidsBatch = data["esearchresult"]["idlist"]
    allePmids.extend(pmidsBatch)
    print(f"{len(allePmids)} / {total}")
    time.sleep(0.1)
allePmids = list(dict.fromkeys(allePmids))

print(f"Fertig - {len(allePmids)} PMIDs insgesamt")

500 / 6505
1000 / 6505
1500 / 6505
2000 / 6505
2500 / 6505
3000 / 6505
3500 / 6505
4000 / 6505
4500 / 6505
5000 / 6505
5500 / 6505
6000 / 6505
6500 / 6505
6505 / 6505
Fertig - 6505 PMIDs insgesamt


In [41]:
alleArtikel = []
fetchBatchSize = 100

for i in range(0, len(allePmids), fetchBatchSize):
    batch = allePmids[i:i + fetchBatchSize]
    ids = ",".join(batch)

    params = {
        "db" : "pubmed",
        "id" : ids,
        "retmode" : "xml",
        "api_key" : apiKey
    }
    response = requests.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi", params=params)
    artikel = parsePubmedXML(response.text)
    alleArtikel.extend(artikel)
    print(f"{len(alleArtikel)} / {len(allePmids)}")
    time.sleep(0.1)
print(f"Fertig - {len(alleArtikel)} Artikel insgesamt")

100 / 6505
200 / 6505
300 / 6505
400 / 6505
500 / 6505
600 / 6505
700 / 6505
800 / 6505
900 / 6505
1000 / 6505
1100 / 6505
1200 / 6505
1300 / 6505
1400 / 6505
1500 / 6505
1600 / 6505
1700 / 6505
1800 / 6505
1900 / 6505
2000 / 6505
2100 / 6505
2200 / 6505
2300 / 6505
2400 / 6505
2500 / 6505
2600 / 6505
2700 / 6505
2800 / 6505
2900 / 6505
3000 / 6505
3100 / 6505
3200 / 6505
3300 / 6505
3400 / 6505
3500 / 6505
3600 / 6505
3700 / 6505
3800 / 6505
3900 / 6505
4000 / 6505
4100 / 6505
4200 / 6505
4300 / 6505
4400 / 6505
4500 / 6505
4600 / 6505
4700 / 6505
4800 / 6505
4900 / 6505
5000 / 6505
5100 / 6505
5200 / 6505
5300 / 6505
5400 / 6505
5500 / 6505
5600 / 6505
5700 / 6505
5800 / 6505
5900 / 6505
6000 / 6505
6100 / 6505
6200 / 6505
6300 / 6505
6400 / 6505
6500 / 6505
6505 / 6505
Fertig - 6505 Artikel insgesamt


In [42]:
test = [a for a in alleArtikel if a["title"] == "kein Titel verfügbar"]
print(f"Artikel ohne Title: {len(test)}")

test2= [a for a in alleArtikel if a["title"] == "[Not Available]."]
print(f"Artikel mit [Not Available]: {len(test2)}")

Artikel ohne Title: 0
Artikel mit [Not Available]: 0


In [44]:
df = pl.DataFrame(alleArtikel)

df = df.with_columns([
    pl.col("affiliations").str.replace_all("\xa0", " "),
    pl.col("title").str.replace_all("\xa0", " ")
])

df.write_csv("pubmed_rohdaten_komplett.csv")

print(f"Gespeichert – {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
df.head()
print(df.shape)

Gespeichert – 6505 Zeilen, 6 Spalten
(6505, 6)
